In [1]:

from fast_bert.prediction import BertClassificationPredictor

# ekman_label.csv renamed to labels.csv

MODEL_PATH = './model/model_out'
LABEL_PATH = './model/'
text_predictor = BertClassificationPredictor(
				model_path=MODEL_PATH,
				label_path=LABEL_PATH, # location for labels.csv file
				multi_label=False,
				# model_type='xlnet',
				do_lower_case=False,
				device='cpu') # set cu


In [16]:
import librosa
import keras
import numpy as np

class LivePredictions:
    """
    Main class of the application.
    """

    def __init__(self):
        """
        Init method is used to initialize the main parameters.
        """
        self.path = './model/Emotion_Voice_Detection_Model.h5'
        self.loaded_model = keras.models.load_model(self.path)

    def make_predictions(self,file_path):
        """
        Method to process the files and create your features.
        """
        data, sampling_rate = librosa.load(file_path,res_type='kaiser_fast')
        x = np.mean(librosa.feature.mfcc(y=data, sr=sampling_rate, n_mfcc=40).T, axis=0).reshape(1,40,1)
        predictions = self.loaded_model.predict(x)
        return list(predictions)[0]
        # print( "Prediction is", " ", self.convert_class_to_emotion(predictions))

sound_predictor = LivePredictions()
'''
label_conversion = {'0': 'neutral',
                    '1': 'happy',
                    '2': 'sad',
                    '3': 'angry',
                    '4': 'fearful',
                    '5': 'disgust',
                    '6': 'surprised'}
'''

"\nlabel_conversion = {'0': 'neutral',\n                    '1': 'happy',\n                    '2': 'sad',\n                    '3': 'angry',\n                    '4': 'fearful',\n                    '5': 'disgust',\n                    '6': 'surprised'}\n"

In [3]:
SOUND_FILE_PATH = '../data/emotion-classification-from-audio-files-master/features'

In [19]:
final_infor_result = []
label = []
label_conversion = {'neutral':'neutral','joy': 'happy','sadness': 'sad','anger': 'angry','fear': 'fearful','disgust': 'disgust','surprise': 'surprised'}

In [6]:
from a2w.audio_handler import audio_to_words,AudioToWords
audio = AudioToWords('./a2w/model')

In [20]:
import os
def get_complete_info(path):
    for subdir, dirs, files in os.walk(path):
            for file in files:
                try:
                    file_path = os.path.join(subdir, file)
                    file = int(file[7:8]) - 1
                    if(file <= 1):
                        label.append(0)
                    elif(file > 1):
                        label.append(file - 1)
                    text = audio_to_words(file_path,audio)
                    result = sound_predictor.make_predictions(file_path)
                    labeled_sound_result = {'neutral': result[0], 'happy': result[1], 'sad': result[2], 'angry': result[3], 'fearful': result[4], 'disgust': result[5], 'surprised': result[6]}
                    text_result = text_predictor.predict(text)
                    labeled_text_result = {}
                    for i in text_result:
                        labeled_text_result[label_conversion[i[0]]] = i[1]
                    file_info = {'file_path': file_path, 'label': label[-1], 'text': text, 'sound_vector': labeled_sound_result,  'text_vector': labeled_text_result}
                    final_infor_result.append(file_info)
                # If the file is not valid, skip it
                except ValueError as err:
                    print(err)
                    continue

In [21]:
get_complete_info(SOUND_FILE_PATH)